# 06 — Gates as Unitary Evolution

Meet the operations that *move* a qubit, and discover why keeping every probability intact forces every one of them to be unitary.

**Prerequisites:** `01/05_inner_product_and_dirac` (bras, overlaps, the Born rule as a squared overlap), `01/04_measurement_and_born_rule`, `01/03_the_bloch_sphere`.

**What you'll be able to do**
- Apply a gate as a matrix acting on a state, $|\psi'\rangle = U|\psi\rangle$, by hand and in code, and read where $|0\rangle$ and $|1\rangle$ go off the columns of $U$.
- Form the dagger $U^\dagger$ of any matrix and use the rule that it reverses products.
- Derive from the inner product of `01/05` why a gate must be **unitary** ($U^\dagger U = I$), read it as "orthonormal columns", and check it with `is_unitary`.
- Prove that a *linear* map keeping every norm keeps every overlap, and break the rule with three counter-examples.
- Set the gate beside its classical cousin, the permutation.

So far we have *described* states (`01/01`–`01/03`), *measured* them (`01/04`) and *compared* them (`01/05`). To compute, we need to **change** them. Every such operation is a quantum gate, and today you learn the one rule they all obey, straight from the inner product you built in `01/05`. What that rule buys you is the subject of `01/07`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from qot_course import viz
from qot_course.colors import COLORS, CMAP_DENSITY, CMAP_PLAN
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, qubit_state
from qot_course.quantum.gates import apply_gate, is_unitary

rng = np.random.default_rng(0)   # random pairs of states in sections 3 and 4, and in the exercises
viz.use_course_style()

## 1. A gate is a matrix acting on a state

A single-qubit gate is a $2\times2$ matrix $U$ with complex entries $U_{ij}$. Applying it to a state $|\psi\rangle = a_0|0\rangle + a_1|1\rangle$ means the **matrix–vector product** — something new for us: until now we only multiplied amplitudes term by term. Written out, each output amplitude is a two-term sum of one *row* of $U$ against the ket:

$$ U|\psi\rangle = \begin{pmatrix} U_{00} & U_{01} \\ U_{10} & U_{11} \end{pmatrix}\begin{pmatrix} a_0 \\ a_1 \end{pmatrix} = \begin{pmatrix} U_{00}\,a_0 + U_{01}\,a_1 \\ U_{10}\,a_0 + U_{11}\,a_1 \end{pmatrix}. $$

Two consequences you can read straight off this formula: $U|0\rangle$ (take $a_0=1, a_1=0$) is the **first column** of $U$, and $U|1\rangle$ is the **second column**. So a $2\times2$ matrix is a compact way of saying "here is where $|0\rangle$ goes, and here is where $|1\rangle$ goes"; the rest follows by linearity, $U(a_0|0\rangle + a_1|1\rangle) = a_0\,U|0\rangle + a_1\,U|1\rangle$. Keep this in your pocket — it is how you will read the named gates of `01/08`–`01/10` at a glance. Applying $U$ first and then $V$ sends $|\psi\rangle$ to $V(U|\psi\rangle) = (VU)|\psi\rangle$: a sequence of gates is the product of their matrices, the gate applied first standing on the right.

Here is a concrete gate, still unnamed — a matrix that tips $|0\rangle$ partway toward $|1\rangle$. With $\theta = 2\pi/3$:

$$ U = \begin{pmatrix} \cos\tfrac{\theta}{2} & -\sin\tfrac{\theta}{2} \\ \sin\tfrac{\theta}{2} & \cos\tfrac{\theta}{2} \end{pmatrix} = \begin{pmatrix} 0.5 & -0.866 \\ 0.866 & 0.5 \end{pmatrix}, \qquad U|0\rangle = \begin{pmatrix} 0.5\cdot 1 + (-0.866)\cdot 0 \\ 0.866\cdot 1 + 0.5\cdot 0 \end{pmatrix} = \begin{pmatrix} 0.5 \\ 0.866 \end{pmatrix}. $$

On $|+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$, linearity says: add the two columns and divide by $\sqrt2$, which gives $(-0.259,\ 0.966)$.

Let's compute it by hand, hand the same job to `apply_gate`, and draw what the matrix does to the Born probabilities of three inputs. For contrast, the figure also shows a matrix that is *not* a gate, the shear $M = \begin{pmatrix}1 & 1\\ 0 & 1\end{pmatrix}$, acting on $|+\rangle$.

In [ ]:
theta = 2 * np.pi / 3
U = np.array([[np.cos(theta / 2), -np.sin(theta / 2)],
              [np.sin(theta / 2),  np.cos(theta / 2)]], dtype=complex)
a0, a1 = KET_0

by_hand = np.array([U[0, 0] * a0 + U[0, 1] * a1,      # first output amplitude: row 0 against the ket
                    U[1, 0] * a0 + U[1, 1] * a1])     # second output amplitude: row 1 against the ket
out = apply_gate(U, KET_0)

print("U =\n", np.round(U, 3))
print("U|0> by hand (row . ket) =", np.round(by_hand, 3))
print("U|0> via apply_gate      =", np.round(out, 3), "  identical:", np.allclose(by_hand, out))
print("first column of U        =", np.round(U[:, 0], 3), "  (= U|0>, as promised)")
print("U|1> = second column     =", np.round(apply_gate(U, KET_1), 3))
print("output norm              =", round(float(np.linalg.norm(out)), 6), " (still 1)")

U_plus = apply_gate(U, KET_PLUS)
print("\nU|+> = (col 0 + col 1)/sqrt2 =", np.round((U[:, 0] + U[:, 1]) / np.sqrt(2), 3),
      "  apply_gate:", np.round(U_plus, 3))
print("Born probabilities of U|+>   =", np.round(np.abs(U_plus) ** 2, 3))

M = np.array([[1.0, 1.0], [0.0, 1.0]], dtype=complex)    # a shear: a matrix, but (section 3) not a gate
M_plus = apply_gate(M, KET_PLUS)
print("M|+> =", np.round(M_plus, 3), "  |amplitudes|^2 =", np.round(np.abs(M_plus) ** 2, 3),
      "  sum =", round(float(np.sum(np.abs(M_plus) ** 2)), 3))

# Figure: |a0|^2 (dark block) stacked under |a1|^2 (light block), before and after each matrix
cases = [(r"$|0\rangle$", KET_0, U, "U"), (r"$|1\rangle$", KET_1, U, "U"),
         (r"$|+\rangle$", KET_PLUS, U, "U"), (r"$|+\rangle$", KET_PLUS, M, "M")]
fig, ax = plt.subplots(figsize=(10.5, 4.4))
ticks, labels = [], []
for k, (lbl, ket, G, gname) in enumerate(cases):
    after_color = COLORS["quantum"] if gname == "U" else COLORS["negative"]
    for j, (vec, tag, col) in enumerate([(ket, "before", COLORS["source"]), (apply_gate(G, ket), f"after {gname}", after_color)]):
        p = np.abs(vec) ** 2
        x = 3 * k + j
        ax.bar(x, p[0], color=col, width=0.8)
        ax.bar(x, p[1], bottom=p[0], color=col, alpha=0.45, width=0.8)
        ax.text(x, p.sum() + 0.05, f"{p.sum():.2f}", ha="center", fontsize=10)
        ticks.append(x); labels.append(f"{lbl}\n{tag}")
ax.axhline(1, color=COLORS["muted"], ls="--", lw=1.2)
ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 2.9); ax.set_ylabel(r"$|a_0|^2$ (dark) $+\ |a_1|^2$ (light)")
ax.set_title("Total probability: U keeps it at 1, the shear M does not")
ax.legend(handles=[Patch(color=COLORS["source"], label="input state"),
                   Patch(color=COLORS["quantum"], label="after the gate U"),
                   Patch(color=COLORS["negative"], label="after the shear M (not a gate)")],
          loc="upper left", fontsize=9)
plt.show()

**Read the output and the figure.** The two-term sums reproduce `apply_gate` exactly, and $U|0\rangle$ is indeed the first column of $U$. The matrix turned the sharp state $|0\rangle$ into a superposition $0.5\,|0\rangle + 0.866\,|1\rangle$ — Born probabilities $0.25$ and $0.75$ — and the output still has norm $1$. On $|+\rangle$, adding the two columns agrees with `apply_gate`, with probabilities $0.067$ and $0.933$.

In the figure, each pair of bars is one input before and after a matrix; the dark block is $|a_0|^2$, the light block $|a_1|^2$. Under $U$ the blocks trade height, but every bar ends exactly on the dashed line at $1$. The shear's bar does not: $M|+\rangle = (1.414, 0.707)$ would give "probabilities" summing to $2.5$.

A state must stay normalised after *any* gate, or the Born-rule probabilities of `01/04` would no longer sum to $1$. That single requirement, taken for *every* state, is what pins down the allowed matrices — but to write it down we first need the dagger of a matrix.

## 2. The dagger of a matrix

In `01/05` you built a bra from a ket by taking the **conjugate transpose**: a column of amplitudes became a row of conjugated amplitudes, $\langle\psi| = |\psi\rangle^\dagger$. The very same operation makes sense for a matrix: **transpose it (swap rows and columns), then conjugate every entry.** The result is written $U^\dagger$ ("U-dagger"), and entry by entry $(U^\dagger)_{ij} = \overline{U_{ji}}$:

$$ U = \begin{pmatrix} U_{00} & U_{01} \\ U_{10} & U_{11} \end{pmatrix} \quad\Longrightarrow\quad U^\dagger = \begin{pmatrix} \overline{U_{00}} & \overline{U_{10}} \\ \overline{U_{01}} & \overline{U_{11}} \end{pmatrix}. $$

Our example $U$ has real entries, so conjugation does nothing and $U^\dagger$ is the plain transpose: the off-diagonal entries $-0.866$ and $0.866$ swap places.

To watch the conjugation at work, take a complex matrix (the dagger is defined for every matrix, gate or not):

$$ A = \begin{pmatrix} 1+2i & 3 \\ -i & 2-i \end{pmatrix} \;\xrightarrow{\ \text{transpose}\ }\; \begin{pmatrix} 1+2i & -i \\ 3 & 2-i \end{pmatrix} \;\xrightarrow{\ \text{conjugate}\ }\; A^\dagger = \begin{pmatrix} 1-2i & i \\ 3 & 2+i \end{pmatrix}. $$

The off-diagonal entries crossed the diagonal, and every imaginary part changed sign. Doing the dagger twice undoes both steps: $(A^\dagger)^\dagger = A$.

The dagger has one rule you will use constantly: **it reverses the order of a product.** The bra of the *evolved* state $U|\varphi\rangle$ is $\langle\varphi|U^\dagger$, not $U^\dagger\langle\varphi|$:

$$ \big(U|\varphi\rangle\big)^\dagger = \langle\varphi|\,U^\dagger . $$

Why: transposing a product reverses it, $(AB)^T = B^T A^T$ (the rows of $A$ become columns, so they must now be met from the other side), and conjugation acts entrywise, so $(AB)^\dagger = B^\dagger A^\dagger$. Let's see both facts on numbers, and draw $A$ beside $A^\dagger$.

In [ ]:
U_dag = U.conj().T                                  # transpose, then conjugate every entry
print("U^dagger =\n", np.round(U_dag, 3) + 0)   # "+ 0" only tidies a cosmetic -0.j
print("entry check (U^dagger)_01 == conj(U_10):", np.isclose(U_dag[0, 1], np.conj(U[1, 0])))

# The bra of the evolved state: (U|phi>)^dagger  vs  <phi| U^dagger  (order reversed)
phi = KET_PLUS
lhs = apply_gate(U, phi).conj()          # bra of U|phi>: conjugate the evolved column
rhs = phi.conj() @ U_dag                 # <phi| U^dagger: bra first, then U^dagger
print("(U|+>)^dagger        =", np.round(lhs, 3) + 0)
print("<+| U^dagger         =", np.round(rhs, 3), "  same:", np.allclose(lhs, rhs))
wrong = U_dag @ phi.conj()               # the un-reversed order gives a different vector
print("U^dagger <+| (wrong) =", np.round(wrong, 3), "  same:", np.allclose(lhs, wrong))

# A complex matrix, where the conjugation step shows
A = np.array([[1 + 2j, 3], [-1j, 2 - 1j]])
A_dag = A.conj().T
print("\nA^dagger =\n", A_dag + 0)
print("matches the hand result [[1-2i, i], [3, 2+i]]:", np.allclose(A_dag, [[1 - 2j, 1j], [3, 2 + 1j]]))
print("dagger twice gives A back:", np.allclose(A_dag.conj().T, A))
B = np.array([[0, 1j], [2, 1]])           # any second matrix, to test (AB)^dagger = B^dagger A^dagger
print("(AB)^dagger == B^dagger A^dagger:", np.allclose((A @ B).conj().T, B.conj().T @ A_dag),
      "   == A^dagger B^dagger (wrong order):", np.allclose((A @ B).conj().T, A_dag @ B.conj().T))

fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.4))
panels = [("Re A", A.real), ("Im A", A.imag), (r"Re A$^\dagger$", A_dag.real), (r"Im A$^\dagger$", A_dag.imag)]
for ax, (name, vals) in zip(axes, panels):
    ax.imshow(vals, cmap=CMAP_DENSITY, vmin=-3, vmax=3)
    for (i, j), v in np.ndenumerate(vals):
        ax.text(j, i, f"{v + 0.0:+.0f}", ha="center", va="center", fontsize=13)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.grid(False)
    ax.set_xlabel("column j"); ax.set_ylabel("row i"); ax.set_title(name)
fig.suptitle("The dagger: transpose (entries cross the diagonal), then conjugate (imaginary parts flip sign)", fontsize=11)
plt.tight_layout()
plt.show()

**Read the output and the figure.** $U^\dagger$ is $U$ with the off-diagonal entries swapped (real matrix, so no conjugation to see — the `entry check` line confirms the definition anyway). The bra of $U|+\rangle$ equals $\langle +|U^\dagger$ with the dagger *after* the bra; putting it in front gives a different vector. This reversal is the one step in the coming derivation that people trip on, so it is worth having watched it fail once. The complex matrix confirms the hand computation, and the product rule holds only in the reversed order.

In the figure, the real parts (first and third panels) are transposes of each other: the $3$ moves from top-right to bottom-left, and the diagonal stays. The imaginary parts (second and fourth panels) are transposed *and* negated: every coloured square changes colour, teal to coral and coral to teal, the conjugation made visible. A real matrix has an empty imaginary panel, which is why our $U$ hid this step.

## 3. Why gates are unitary

**The requirement.** In `01/05` §2 you saw that the Born rule is a squared overlap, $P = |\langle x|\psi\rangle|^2$, where $|x\rangle$ is the state you ask for: so far a basis ket or $|\pm\rangle$ (the recombine-then-measure test of `01/04`), and, as `01/09` will show, any state at all. A gate on an isolated qubit loses nothing: two states that a measurement could tell apart with some probability before the gate must be told apart exactly as well after it. Every such probability is a squared overlap, so the requirement is that the gate keeps every squared overlap, $|\langle U\varphi|U\psi\rangle|^2 = |\langle\varphi|\psi\rangle|^2$. This is the assumption of a closed system, and section 4 shows that a weaker one, keeping every norm, already suffices. A gate is also *linear*, and for a linear map keeping every squared overlap is the same as keeping every overlap itself, phase included (section 4 proves it). (Drop linearity, and, up to a phase that may differ from state to state, the one other kind of map that keeps every squared overlap is *anti*-unitary: it also conjugates the amplitudes. That is Wigner's theorem, proved with care by Bargmann (1964), and no gate is of that kind; section 4 shows the simplest example.) Hence the rule: **a gate must preserve the inner product of every pair of states,**

$$ \langle U\varphi\,|\,U\psi\rangle = \langle\varphi|\psi\rangle \quad\text{for all }|\varphi\rangle,|\psi\rangle. $$

**The derivation.** Section 2 gives us the left-hand side: the bra of $U|\varphi\rangle$ is $\langle\varphi|U^\dagger$, so

$$ \langle U\varphi\,|\,U\psi\rangle = \langle\varphi|\,U^\dagger U\,|\psi\rangle . $$

The requirement therefore reads $\langle\varphi|\,U^\dagger U\,|\psi\rangle = \langle\varphi|\psi\rangle$ for all pairs. Now use the trick of `01/05` §3: choose $|\varphi\rangle,|\psi\rangle$ among the basis kets $|0\rangle,|1\rangle$. For any matrix $A$, $\langle i|A|j\rangle$ picks out the entry $A_{ij}$ (row $i$, column $j$), while $\langle i|j\rangle$ is $1$ if $i=j$ and $0$ otherwise. So the four choices $(i,j)\in\{0,1\}^2$ read off the four entries of $U^\dagger U$ one at a time and force

$$ U^\dagger U = \begin{pmatrix} 1 & 0 \\ 0 & 1\end{pmatrix} = I . $$

Conversely, if $U^\dagger U = I$ then every overlap is preserved by the display above. A matrix with $U^\dagger U = I$ is called **unitary** — so "valid gate" and "unitary matrix" name the same thing (up to one small proviso, a global phase, that `01/07` makes precise).

**A second reading: orthonormal columns.** Row $i$ of $U^\dagger$ is the bra of column $i$ of $U$, so $(U^\dagger U)_{ij} = \langle u_i|u_j\rangle$, the overlap of columns $i$ and $j$. Hence $U^\dagger U = I$ says: **each column has norm $1$, and the two columns are orthogonal.** The columns are $U|0\rangle$ and $U|1\rangle$ (section 1): a gate sends the orthonormal pair $|0\rangle, |1\rangle$ to another orthonormal pair.

**By hand on our $U$.** Multiply $U^\dagger$ (rows) against $U$ (columns), four two-term sums:

$$ (U^\dagger U)_{00} = \cos^2\tfrac{\theta}{2} + \sin^2\tfrac{\theta}{2} = 1, \qquad (U^\dagger U)_{01} = -\cos\tfrac{\theta}{2}\sin\tfrac{\theta}{2} + \sin\tfrac{\theta}{2}\cos\tfrac{\theta}{2} = 0, $$

and the other two entries the same way. The shear's columns $(1, 0)$ and $(1, 1)$ fail both tests: the second has squared norm $2$, and they overlap by $1$. Let's print those sums, confirm with `is_unitary`, and test the requirement itself on hundreds of random pairs.

In [ ]:
c, s = np.cos(theta / 2), np.sin(theta / 2)
print("(U^dag U)_00 by hand = c^2 + s^2      =", round(c * c + s * s, 6))
print("(U^dag U)_01 by hand = -c*s + s*c     =", round(-c * s + s * c, 6))
print("(U^dag U)_10 by hand = -s*c + c*s     =", round(-s * c + c * s, 6))
print("(U^dag U)_11 by hand = s^2 + c^2      =", round(s * s + c * c, 6))
print("U^dagger U (numpy) =\n", np.round(U_dag @ U, 6) + 0)
print("U U^dagger (numpy) =\n", np.round(U @ U_dag, 6) + 0, "  (for a square matrix both orders agree; is_unitary checks this one)")
print("U is unitary?      ", is_unitary(U))
print("columns of U: squared norms", np.round([np.vdot(U[:, 0], U[:, 0]).real, np.vdot(U[:, 1], U[:, 1]).real], 6),
      "  overlap <u0|u1> =", np.round(np.vdot(U[:, 0], U[:, 1]), 6) + 0)

# The overlap of a random pair survives U (the requirement, tested directly)
psi_a, psi_b = (v / np.linalg.norm(v) for v in rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2)))
print("\n<a|b>       =", np.round(np.vdot(psi_a, psi_b), 6))
print("<Ua|Ub>     =", np.round(np.vdot(apply_gate(U, psi_a), apply_gate(U, psi_b)), 6), " (unchanged)")

# The shear: it does NOT preserve the norm, so it is not a valid gate.
print("\nM is unitary?      ", is_unitary(M))
print("M^dagger M =\n", np.round(M.conj().T @ M, 3).real, "  (not the identity: column 1 has squared norm 2, the columns overlap by 1)")
print("norm before M       :", round(float(np.linalg.norm(KET_PLUS)), 3))
print("norm after  M       :", round(float(np.linalg.norm(M_plus)), 3), " = sqrt(5/2) =", round(np.sqrt(5 / 2), 3), " (!= 1: not a valid gate)")

# Many pairs at once: squared overlap before vs after, for U and for M
def random_states(n):
    '''n random normalised qubit states (rows), from complex Gaussian amplitudes.'''
    v = rng.normal(size=(n, 2)) + 1j * rng.normal(size=(n, 2))
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def sq_overlaps(G, phis, psis):
    '''|<G phi|G psi>|^2 for each row pair (rows are states, so G acts as row @ G.T).'''
    return np.abs(np.sum((phis @ G.T).conj() * (psis @ G.T), axis=1)) ** 2

phis, psis = random_states(300), random_states(300)
before = sq_overlaps(np.eye(2), phis, psis)
after_U, after_M = sq_overlaps(U, phis, psis), sq_overlaps(M, phis, psis)
print(f"\n300 random pairs: largest change of |<phi|psi>|^2 under U = {np.max(np.abs(after_U - before)):.1e}")
print(f"under M the 'squared overlaps' range from {after_M.min():.3f} to {after_M.max():.3f}  (a probability cannot exceed 1)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"wspace": 0.28})
for ax, after, name, col in [(axes[0], after_U, "the gate U", COLORS["quantum"]), (axes[1], after_M, "the shear M", COLORS["negative"])]:
    lim = max(1.0, after.max()) * 1.05
    ax.plot([0, lim], [0, lim], color=COLORS["muted"], ls="--", lw=1.2, label="unchanged (diagonal)")
    ax.axhline(1, color=COLORS["grid"], lw=1.5)
    ax.scatter(before, after, s=12, color=col, alpha=0.8, label="one random pair")
    ax.set_xlim(0, 1.05); ax.set_ylim(0, lim)
    ax.set_xlabel(r"$|\langle\varphi|\psi\rangle|^2$ before"); ax.set_ylabel("the same quantity after")
    ax.set_title(f"300 random pairs through {name}"); ax.legend(loc="upper left", fontsize=9)
plt.show()

**Read the output and the figure.** The four two-term sums give $1, 0, 0, 1$: $U^\dagger U = I$ by hand, confirmed by NumPy in both orders (for a square matrix $U^\dagger U = I$ and $UU^\dagger = I$ are equivalent, which is why `is_unitary` may check either). The columns are two unit vectors with overlap $0$. A random pair of states keeps its overlap through $U$ — the requirement itself, tested directly. The shear $M$ fails: $M^\dagger M \neq I$, and it stretches the normalised $|+\rangle$ to norm $\sqrt{5/2}\approx 1.58$.

In the figure, each dot is one random pair of states. On the left, every dot sits on the diagonal: $U$ left each squared overlap where it was (the largest change over $300$ pairs is a rounding error, $7\times10^{-16}$). On the right, the dots scatter far from the diagonal, up to $6.3$, far above the grey line at $1$ where probabilities must stay.

For an isolated qubit (a **closed system**, nothing leaking in or out) such a map cannot be an evolution. Unitarity is the gatekeeper. Hold on to the qualifier "closed": when a qubit talks to its environment, non-unitary maps do appear — that is the story of `01/20_quantum_channels`, and unitarity is exactly the ideal it departs from.

## 4. Linear and norm-preserving is enough

Section 3 promised that, for a *linear* map, keeping every norm already implies keeping every overlap. Seeing why shows which ingredient does the work.

**By hand: an overlap from four norms.** Expand a squared norm with $\|v\|^2 = \langle v|v\rangle$, using that $\langle\psi|\varphi\rangle$ is the conjugate of $\langle\varphi|\psi\rangle$ (swap the two states in the sum $b_0^*a_0 + b_1^*a_1$ of `01/05` §1 and every term is conjugated):

$$ \|\varphi + \psi\|^2 = \|\varphi\|^2 + \|\psi\|^2 + \langle\varphi|\psi\rangle + \overline{\langle\varphi|\psi\rangle} = \|\varphi\|^2 + \|\psi\|^2 + 2\,\mathrm{Re}\,\langle\varphi|\psi\rangle . $$

Now replace $\psi$ by $i\psi$. The overlap becomes $\langle\varphi|i\psi\rangle = i\,\langle\varphi|\psi\rangle$, and the real part of $i z$ is $-\mathrm{Im}\,z$, so

$$ \|\varphi + i\psi\|^2 = \|\varphi\|^2 + \|\psi\|^2 - 2\,\mathrm{Im}\,\langle\varphi|\psi\rangle . $$

So four norms, those of $\varphi$, $\psi$, $\varphi + \psi$ and $\varphi + i\psi$, fix the real and imaginary parts of the overlap: the **polarisation identity**.

Now let $U$ be linear and keep every norm. Linearity gives $U(\varphi + \psi) = U\varphi + U\psi$ and $U(\varphi + i\psi) = U\varphi + i\,U\psi$, so the same two lines hold for $U\varphi, U\psi$ with all four norms unchanged. Hence $\langle U\varphi|U\psi\rangle = \langle\varphi|\psi\rangle$, and section 3 forces $U^\dagger U = I$. ($\varphi + \psi$ is not normalised, but $U(c\,v) = c\,Uv$, so a linear map that keeps the norm of every state keeps the norm of every vector.)

**Where linearity entered.** In splitting $U(\varphi + \psi)$, and in pulling the factor $i$ out of $U(i\psi)$. The second step can fail. **Complex conjugation** $K(a_0, a_1) = (\bar a_0, \bar a_1)$ keeps every norm and every squared overlap, since $\langle K\varphi|K\psi\rangle = \overline{\langle\varphi|\psi\rangle}$. Yet $K(i\psi) = -i\,K\psi$: $K$ is not linear, no matrix does it, and it conjugates each overlap. It is the anti-unitary kind of section 3's remark; `01/07` shows what it does to the Bloch sphere.

**Checking the basis kets is not enough.** It is tempting to check only that each column is a unit vector. This matrix passes that test and is still no gate:

$$ N = \begin{pmatrix} 1 & \tfrac{1}{\sqrt2} \\ 0 & \tfrac{1}{\sqrt2} \end{pmatrix}, \qquad N|0\rangle = |0\rangle, \qquad N|1\rangle = |+\rangle . $$

Both columns are unit vectors, but $\langle 0|+\rangle = \tfrac{1}{\sqrt2}$. On $|+\rangle$ that cross term appears: $\|N|+\rangle\|^2 = \tfrac12\big(1 + 1 + 2\cdot\tfrac{1}{\sqrt2}\big) = 1 + \tfrac{1}{\sqrt2} \approx 1.707$, and on $|-\rangle$, with a minus sign, $1 - \tfrac{1}{\sqrt2} \approx 0.293$. Orthogonal columns are part of unitarity, not an extra.

Let's check the identity on a random pair, run the counter-examples, and sweep $\|G\psi\|^2$ for each map $G$ along two circles of states built with `qubit_state(theta, phi)` of `01/03`.

In [ ]:
# The polarisation identity on a random pair: Re and Im of <phi|psi> from four norms
phi_r, psi_r = random_states(2)
n2 = lambda v: np.vdot(v, v).real
re_from_norms = (n2(phi_r + psi_r) - n2(phi_r) - n2(psi_r)) / 2
im_from_norms = -(n2(phi_r + 1j * psi_r) - n2(phi_r) - n2(psi_r)) / 2
print("<phi|psi>               =", np.round(np.vdot(phi_r, psi_r), 6))
print("from the four norms     =", np.round(re_from_norms + 1j * im_from_norms, 6))

# Counter-example 1: N has unit columns that are not orthogonal
N = np.array([[1.0, 1 / np.sqrt(2)], [0.0, 1 / np.sqrt(2)]], dtype=complex)
print("\nN is unitary?", is_unitary(N), "  column norms:", np.round(np.linalg.norm(N, axis=0), 6),
      "  column overlap:", round(float(np.vdot(N[:, 0], N[:, 1]).real), 6))
print("||N|+>||^2 =", round(n2(apply_gate(N, KET_PLUS)), 6), "  1 + 1/sqrt2 =", round(1 + 1 / np.sqrt(2), 6))
print("||N|->||^2 =", round(n2(apply_gate(N, qubit_state(np.pi / 2, np.pi))), 6), "  1 - 1/sqrt2 =", round(1 - 1 / np.sqrt(2), 6))

# Counter-example 2: complex conjugation K keeps norms and squared overlaps, but is not linear
K = lambda v: np.conj(v)
print("\nK: norm of psi before / after       :", round(np.sqrt(n2(psi_r)), 6), "/", round(np.sqrt(n2(K(psi_r))), 6))
print("K: |<phi|psi>|^2 before / after     :", round(abs(np.vdot(phi_r, psi_r)) ** 2, 6), "/", round(abs(np.vdot(K(phi_r), K(psi_r))) ** 2, 6))
print("K: <phi|psi> before / after         :", np.round(np.vdot(phi_r, psi_r), 6), "/", np.round(np.vdot(K(phi_r), K(psi_r)), 6), " (conjugated)")
print("K(i psi) == i K(psi)?               :", np.allclose(K(1j * psi_r), 1j * K(psi_r)),
      "   K(i psi) == -i K(psi)?", np.allclose(K(1j * psi_r), -1j * K(psi_r)))

# Sweep the squared norm of G psi along two great circles of states
thetas = np.linspace(0, np.pi, 181)
phis_eq = np.linspace(0, 2 * np.pi, 361)
shear_mer = np.array([n2(M @ qubit_state(t, 0.0)) for t in thetas])
shear_eq = np.array([n2(M @ qubit_state(np.pi / 2, p)) for p in phis_eq])
print(f"\nshear on the meridian: {shear_mer[90]:.3f} at |+>, peak {shear_mer.max():.3f} at theta = {thetas[shear_mer.argmax()] / np.pi:.2f} pi,"
      f" {shear_mer[-1]:.3f} at |1>;  (3 + sqrt 5)/2 = {(3 + np.sqrt(5)) / 2:.3f}")
print(f"shear on the equator: from {shear_eq.max():.3f} at |+> down to {shear_eq.min():.3f} at |->")
maps = [("U (a gate)", lambda v: U @ v, COLORS["quantum"], "-"),
        ("M (shear)", lambda v: M @ v, COLORS["negative"], "-"),
        ("N (unit, non-orthogonal columns)", lambda v: N @ v, COLORS["target"], "-"),
        ("K (conjugation, not linear)", K, COLORS["highlight"], "--")]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"wspace": 0.25})
for name, G, col, ls in maps:
    axes[0].plot(thetas / np.pi, [n2(G(qubit_state(t, 0.0))) for t in thetas], color=col, ls=ls, lw=2.2, label=name)
    axes[1].plot(phis_eq / np.pi, [n2(G(qubit_state(np.pi / 2, p))) for p in phis_eq], color=col, ls=ls, lw=2.2, label=name)
axes[0].set_xlabel(r"$\theta/\pi$  (state $\cos\frac{\theta}{2}|0\rangle + \sin\frac{\theta}{2}|1\rangle$)")
axes[1].set_xlabel(r"$\phi/\pi$  (state $\frac{1}{\sqrt{2}}(|0\rangle + e^{i\phi}|1\rangle)$)")
for ax, title in [(axes[0], r"From $|0\rangle$ through $|+\rangle$ to $|1\rangle$"), (axes[1], r"Around the equator, from $|+\rangle$ through $|-\rangle$")]:
    ax.set_ylabel(r"$\|G\psi\|^2$  (sum of the would-be probabilities)"); ax.set_ylim(0, 2.7); ax.set_title(title)
axes[0].legend(loc="upper left", fontsize=9)
plt.show()

**Read the output and the figure.** The four norms rebuild the overlap to six decimals; $N$ gives $1.707$ on $|+\rangle$ and $0.293$ on $|-\rangle$, as computed by hand; conjugation keeps norm and squared overlap but conjugates the overlap and pulls $i$ out as $-i$.

In the figure, only the gate $U$ and the conjugation $K$ keep the total $|a_0|^2 + |a_1|^2$ flat on $1$. The shear passes $2.5$ at $|+\rangle$, peaks near $2.62$ a little beyond it and ends at $2$ on $|1\rangle$; around the equator it swings from $2.5$ at $|+\rangle$ down to $0.5$ at $|-\rangle$. $N$ is the instructive one: at the two basis kets (both ends of the left panel) it sits exactly on $1$, but in between it rises to $1.707$, and around the equator it swings between $1.707$ and $0.293$. And $K$, which passes every norm test, fails linearity. A gate needs both properties, and together they are exactly unitarity.

## 5. The classical cousin: permutations

Ask the same question of a classical bit, a probability vector $p = (p_0, p_1)$. A matrix keeping every such vector a probability vector has non-negative entries and columns summing to $1$: a **stochastic** matrix (`01/20` gives it its full role). The classical counterpart of "keeps every overlap" is "keeps every pair of distributions as far apart as before", with the distance $\|p - q\|_1 = \sum_x |p_x - q_x|$.

**By hand.** Every stochastic $2\times2$ matrix has the form

$$ T = \begin{pmatrix} t & s \\ 1-t & 1-s \end{pmatrix}, \qquad t, s \in [0, 1]. $$

Take $p = (a, 1-a)$ and $q = (b, 1-b)$. Then $Tp - Tq = (a - b)\,\big(t - s,\ -(t - s)\big)$, so

$$ \|Tp - Tq\|_1 = |t - s|\;\|p - q\|_1 . $$

Every distance shrinks by the same factor $|t - s| \le 1$, and all are kept only when $|t - s| = 1$: the identity ($t = 1, s = 0$) and the **NOT** $\begin{pmatrix}0 & 1\\ 1 & 0\end{pmatrix}$ ($t = 0, s = 1$). These are the two **permutations** of $\{0, 1\}$. A permutation matrix has a single $1$ in each row and column, so its columns are orthonormal and it is unitary too: the reversible operations on a bit sit inside the quantum gates.

Every other stochastic matrix merges distributions at each step. For $T = \begin{pmatrix}0.9 & 0.2\\ 0.1 & 0.8\end{pmatrix}$ the factor is $0.7$, so the sharp inputs $(1, 0)$ and $(0, 1)$, at distance $2$, come out at $1.4$, then $0.98$. Its inverse matrix, $\tfrac{1}{0.7}\begin{pmatrix}0.8 & -0.2\\ -0.1 & 0.9\end{pmatrix}$, has negative entries, so no stochastic matrix undoes it. Let's draw the factor over all stochastic matrices and follow the distance step by step.

In [ ]:
def stochastic(t, s):
    '''The 2x2 stochastic matrix with first row (t, s): column j is the output law for input j.'''
    return np.array([[t, s], [1 - t, 1 - s]])

l1 = lambda u, v: float(np.sum(np.abs(u - v)))
p, q = np.array([1.0, 0.0]), np.array([0.0, 1.0])
T, NOT = stochastic(0.9, 0.2), stochastic(0.0, 1.0)
print("factor |t - s| of T            :", round(abs(T[0, 0] - T[0, 1]), 3))
for k in range(4):
    Tk, NOTk = np.linalg.matrix_power(T, k), np.linalg.matrix_power(NOT, k)
    print(f"after {k} steps: ||T^k p - T^k q||_1 = {l1(Tk @ p, Tk @ q):.4f}    with NOT: {l1(NOTk @ p, NOTk @ q):.1f}")
print("T^-1 =\n", np.round(np.linalg.inv(T), 4), "  (negative entries: not stochastic)")
print("NOT is unitary?", is_unitary(NOT), "   identity is unitary?", is_unitary(stochastic(1.0, 0.0)), "   T is unitary?", is_unitary(T))

# A random check of the factor rule on random stochastic matrices and random pairs of distributions
ok = True
for _ in range(200):
    t, s, a, b = rng.uniform(size=4)
    Tr = stochastic(t, s)
    pa, pb = np.array([a, 1 - a]), np.array([b, 1 - b])
    ok &= np.isclose(l1(Tr @ pa, Tr @ pb), abs(t - s) * l1(pa, pb))
print("factor rule holds on 200 random (T, p, q):", bool(ok))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"wspace": 0.3})
grid = np.linspace(0, 1, 201)
tt, ss = np.meshgrid(grid, grid)
im = axes[0].imshow(np.abs(tt - ss), origin="lower", extent=(0, 1, 0, 1), cmap=CMAP_PLAN, vmin=0, vmax=1)
fig.colorbar(im, ax=axes[0], label=r"shrinking factor $|t - s|$")
for (tx, sx, lbl, ha) in [(1, 0, "identity", "right"), (0, 1, "NOT", "left_low"), (0.9, 0.2, "T", "right")]:
    axes[0].plot(tx, sx, "o", color=COLORS["highlight"], ms=9, clip_on=False)
    axes[0].annotate(lbl, (tx, sx), textcoords="offset points", xytext={"right": (-10, 8), "left": (10, 8), "left_low": (12, -14)}[ha], ha=ha.split("_")[0], fontsize=11)
axes[0].set_xlabel("t  (probability that input 0 stays 0)"); axes[0].set_ylabel("s  (probability that input 1 becomes 0)")
axes[0].set_title("Every stochastic 2x2 matrix, coloured by its factor"); axes[0].grid(False)
steps = np.arange(0, 11)
axes[1].plot(steps, [l1(np.linalg.matrix_power(T, k) @ p, np.linalg.matrix_power(T, k) @ q) for k in steps], "o-", color=COLORS["source"], lw=2, label="T: distance x 0.7 per step")
axes[1].plot(steps, [l1(np.linalg.matrix_power(NOT, k) @ p, np.linalg.matrix_power(NOT, k) @ q) for k in steps], "s-", color=COLORS["flow"], lw=2, label="NOT: distance kept")
axes[1].set_xlabel("number of steps k"); axes[1].set_ylabel(r"$\|T^k p - T^k q\|_1$")
axes[1].set_ylim(0, 2.2); axes[1].set_title(r"Two sharp inputs $(1,0)$ and $(0,1)$, step by step"); axes[1].legend(loc="center right")
plt.show()

**Read the output and the figure.** Under $T$ the distance goes $2, 1.4, 0.98, 0.686$, times $0.7$ per step, while NOT keeps it at $2$. NOT and the identity pass `is_unitary`, $T$ does not, and the factor rule holds on $200$ random cases.

On the left, each point of the square is one stochastic matrix, coloured by its factor. The factor reaches $1$ only at two corners, the identity and NOT; along the diagonal $t = s$ it is $0$, matrices that forget the bit entirely. Our $T$ sits near the identity corner but not on it, and the right panel shows the cost: its distance decays geometrically while NOT stays flat.

One pattern, two worlds: keeping every "how alike" or "how far" score forces unitaries for a qubit and permutations for a bit.

## Your turn

1. **A complex gate, by hand.** Build $U_\varphi = \mathrm{diag}(1,\,e^{i\varphi})$ for $\varphi = \pi/2$. Write $U_\varphi^\dagger$ by hand (conjugation matters here), compute $U_\varphi^\dagger U_\varphi$, and confirm with `is_unitary`. Show that $U_\varphi$ leaves $|a_0|^2, |a_1|^2$ of every state unchanged, yet $|\langle +|U_\varphi|+\rangle|^2 \neq 1$: compute it by hand and in code.
2. **Measurement is not a gate.** A measurement that reports outcome $0$ sends *every* input to $|0\rangle$. Apply the requirement of section 3 to the pair $|0\rangle,|1\rangle$: what should $\langle U0|U1\rangle$ be, and what is it if both outputs are $|0\rangle$? Which column condition does the matrix with both columns equal to $|0\rangle$ break?
3. **Composition.** If $U$ and $V$ are unitary, show that $VU$ is unitary, using $(VU)^\dagger = U^\dagger V^\dagger$ from section 2. Check it in code with $V = U_\varphi$ from exercise 1. (Circuits, in `01/12`, are exactly such products.)
4. **Harder: the classical theorem with three outcomes.** Let a $3\times3$ stochastic matrix $T$ keep $\|p - q\|_1$ for every pair. (a) Using the sharp inputs $e_i, e_j$ ($i \neq j$, distance $2$), show that columns $i$ and $j$ of $T$ share no outcome (distance $2$ means disjoint supports). (b) Conclude that $T$ is one of the six permutation matrices. (c) Draw $200$ random stochastic matrices with `rng` and confirm that each shrinks the distance of some pair of sharp inputs, while the six permutations keep every distance.

## What you built

- You can apply a gate as a matrix–vector product by hand, and read $U|0\rangle$, $U|1\rangle$ off the columns of $U$.
- You can form the dagger of any matrix (transpose, then conjugate) and use $(AB)^\dagger = B^\dagger A^\dagger$, including $(U|\varphi\rangle)^\dagger = \langle\varphi|U^\dagger$.
- You can derive from kept inner products that a gate is **unitary** ($U^\dagger U = I$), read it as "orthonormal columns", check it with `is_unitary`, and say why it holds for a closed system.
- You can prove with the polarisation identity that a linear, norm-keeping map keeps every overlap, and say where the shear, $N$ and conjugation each fail.
- You can show by hand that the stochastic $2\times2$ matrices keeping every distance are the two permutations, which are unitary too.

You can now *operate* on quantum states, not only describe them — a genuine step up. Everything computational the course does is built from these matrices.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| a reversible operation on a bit: a permutation (identity, NOT) | a gate: a unitary matrix $U$, $U^\dagger U = I$ |

## What's next

You know what a gate must be. In `01/07_gates_as_bloch_rotations` you collect what that rule gives you: a global phase that changes nothing, a gate that can always be undone, and a rigid turn of the Bloch sphere.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, chapter 1 "Introduction and overview" and chapter 2 "Introduction to quantum mechanics", Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- J. J. Sakurai & J. Napolitano, *Modern Quantum Mechanics*, chapter 1 "Fundamental Concepts" and chapter 2 "Quantum Dynamics", 3rd ed., Cambridge University Press (2020). doi:10.1017/9781108587280
- V. Bargmann, "Note on Wigner's theorem on symmetry operations", *Journal of Mathematical Physics* 5, 862–868 (1964) — a map that keeps every squared overlap is unitary or anti-unitary. doi:10.1063/1.1704188

**Previous:** `notebooks/01_QuantumFoundations/05_inner_product_and_dirac.ipynb` · **Next:** `notebooks/01_QuantumFoundations/07_gates_as_bloch_rotations.ipynb`